In [ ]:
import randomimport mathfrom analysis_utils import (    polynomial_design,    normal_equation_solve,    predict,    loocv_error,    svg_scatter,    truncated_power_basis,    natural_cubic_spline_basis,    smoothing_spline,    smoothing_spline_predict,    linspace,    subset_metrics,    combinations,    forward_stepwise,    backward_stepwise,    lasso_path_cv,    read_auto_dataset,    backfitting_gam,    svg_line_plot,    polynomial_fit,    evaluate_polynomial,    k_fold_indices)

In [ ]:
# Part I (a)random.seed(634)x = [random.gauss(0, 1) for _ in range(100)]y = [xi - 2 * xi * xi + random.gauss(0, 1) for xi in x]n = len(x)p = 1print(f"n = {n}")print(f"p = {p}")print("Model: Y = X - 2 * X^2 + error")svg_scatter(x, y, filename='partI_a_scatter.svg', title='Part I (a): Simulated Data', xlabel='X', ylabel='Y')print('Saved scatter plot to partI_a_scatter.svg')

In [ ]:
# Part I (b)degrees = [1, 2, 3, 4]loocv_results = {}for d in degrees:    err = loocv_error(x, y, d)    loocv_results[d] = err    print(f"Degree {d}: LOOCV error = {err:.4f}")print("LOOCV results would change with a different random seed because the simulated data would change.")

In [ ]:
# Part I (c)best_degree = min(loocv_results, key=loocv_results.get)print(f"Smallest LOOCV error achieved at degree {best_degree}.")if best_degree == 2:    print("This matches expectations because the true data-generating process is quadratic.")else:    print("This differs from the quadratic truth due to noise and sample variability.")

In [ ]:
# Part I (d)feature_matrix = [[xi ** d for d in range(1, 11)] for xi in x]full_design = [[1.0] + row for row in feature_matrix]full_beta = normal_equation_solve(full_design, y)full_fit = predict(full_design, full_beta)residuals = [yi - fi for yi, fi in zip(y, full_fit)]sigma_full_sq = sum(r * r for r in residuals) / (n - len(full_beta))all_results = []for k in range(1, 11):    for combo in combinations(list(range(10)), k):        metrics = subset_metrics(feature_matrix, y, combo, sigma_full_sq)        all_results.append({            'size': k,            'combo': combo,            'beta': metrics['beta'],            'cp': metrics['cp'],            'bic': metrics['bic'],            'adj_r2': metrics['adj_r2']        })sizes = sorted(set(entry['size'] for entry in all_results))cp_values = []bic_values = []adj_values = []for size in sizes:    candidates = [entry for entry in all_results if entry['size'] == size]    cp_values.append(min(c['cp'] for c in candidates))    bic_values.append(min(c['bic'] for c in candidates))    adj_values.append(max(c['adj_r2'] for c in candidates))svg_line_plot(sizes, [(cp_values, 'C_p', '0')], 'partI_d_cp.svg', 'Part I (d): Mallows C_p', 'Predictor Count', 'C_p')svg_line_plot(sizes, [(bic_values, 'BIC', '0')], 'partI_d_bic.svg', 'Part I (d): BIC', 'Predictor Count', 'BIC')svg_line_plot(sizes, [(adj_values, 'Adjusted R^2', '0')], 'partI_d_adjR2.svg', 'Part I (d): Adjusted R^2', 'Predictor Count', 'Adjusted R^2')print('Saved plots for C_p, BIC, and adjusted R^2.')best_cp = min(all_results, key=lambda entry: entry['cp'])best_bic = min(all_results, key=lambda entry: entry['bic'])best_adj = max(all_results, key=lambda entry: entry['adj_r2'])def describe_solution(tag, result):    names = [f"X^{idx + 1}" for idx in result['combo']]    coef_map = {'Intercept': result['beta'][0]}    for pos, idx in enumerate(result['combo']):        coef_map[names[pos]] = result['beta'][pos + 1]    print(f"Best by {tag}: predictors {names}")    for key, value in coef_map.items():        print(f"  {key}: {value:.4f}")describe_solution('C_p', best_cp)describe_solution('BIC', best_bic)describe_solution('Adjusted R^2', best_adj)

In [ ]:
# Part I (e)forward_history = forward_stepwise(feature_matrix, y, sigma_full_sq)backward_history = backward_stepwise(feature_matrix, y, sigma_full_sq)forward_best = min(forward_history, key=lambda entry: entry['bic'])backward_best = min(backward_history, key=lambda entry: entry['bic'])def summarize_stepwise(name, result):    names = [f"X^{idx + 1}" for idx in result['features']]    print(f"{name} best model (by BIC): predictors {names}")    print(f"  BIC: {result['bic']:.4f}, Adjusted R^2: {result['adj_r2']:.4f}")summarize_stepwise('Forward stepwise', forward_best)summarize_stepwise('Backward stepwise', backward_best)

In [ ]:
# Part I (f)lambda_grid = [0.01, 0.1, 1.0]best_lambda, cv_errors, lasso_coeffs = lasso_path_cv(feature_matrix, y, lambda_grid, k=5)lambda_sequence = sorted(cv_errors.keys())cv_line = [cv_errors[lam] for lam in lambda_sequence]svg_line_plot(lambda_sequence, [(cv_line, 'CV Error', '0')], 'partI_f_lasso_cv.svg', 'Part I (f): Lasso CV Error', 'Lambda', 'Cross-Validation Error')print(f"Best lambda: {best_lambda}")coefficients = lasso_coeffs[best_lambda]for idx, coeff in enumerate(coefficients):    if idx == 0:        print(f"  Intercept: {coeff:.4f}")    else:        print(f"  X^{idx}: {coeff:.4f}")

In [ ]:
# Part II (dataset preparation)auto_data, horsepower_values, mpg_values, horsepower_copy = read_auto_dataset('Auto.csv')print(f"Cleaned observations: {len(auto_data)}")print("First five cleaned rows:")for row in auto_data[:5]:    print(row)

In [ ]:
# Part II (a)degrees = list(range(1, 8))coefficients_per_degree = {}x_grid = linspace(min(horsepower_values), max(horsepower_values), 200)curves = []for degree in degrees:    beta = polynomial_fit(horsepower_values, mpg_values, degree)    coefficients_per_degree[degree] = beta    preds = []    for value in x_grid:        terms = [1.0] + [value ** d for d in range(1, degree + 1)]        preds.append(sum(terms[i] * beta[i] for i in range(len(beta))))    curves.append((x_grid, preds, f"Degree {degree}"))svg_scatter(horsepower_values, mpg_values, filename='partII_a_polynomials.svg', title='Part II (a): Polynomial Fits', xlabel='Horsepower', ylabel='MPG', curves=curves)print('Saved polynomial comparison plot to partII_a_polynomials.svg')

In [ ]:
# Part II (b)folds = k_fold_indices(len(horsepower_values), 10, seed=42)cv_errors_poly = {}for degree in degrees:    errors = []    for fold in folds:        train_idx = [i for i in range(len(horsepower_values)) if i not in fold]        test_idx = fold        train_x = [horsepower_values[i] for i in train_idx]        train_y = [mpg_values[i] for i in train_idx]        beta = polynomial_fit(train_x, train_y, degree)        for idx in test_idx:            value = horsepower_values[idx]            row = [1.0] + [value ** d for d in range(1, degree + 1)]            pred = sum(row[i] * beta[i] for i in range(len(beta)))            errors.append((mpg_values[idx] - pred) ** 2)    cv_errors_poly[degree] = sum(errors) / len(errors)for degree in degrees:    print(f"Degree {degree}: CV error = {cv_errors_poly[degree]:.4f}")best_degree_poly = min(cv_errors_poly, key=cv_errors_poly.get)print(f"Optimal degree by cross-validation: {best_degree_poly}")best_beta = coefficients_per_degree[best_degree_poly]best_curve = []for value in x_grid:    row = [1.0] + [value ** d for d in range(1, best_degree_poly + 1)]    best_curve.append(sum(row[i] * best_beta[i] for i in range(len(best_beta))))svg_scatter(horsepower_values, mpg_values, filename='partII_b_best_poly.svg', title='Part II (b): Optimal Polynomial Fit', xlabel='Horsepower', ylabel='MPG', curves=[(x_grid, best_curve, f"Degree {best_degree_poly}")])print('Saved optimal polynomial fit plot to partII_b_best_poly.svg')

In [ ]:
# Part II (c)sorted_hp = sorted(horsepower_values)quantiles = [sorted_hp[int(len(sorted_hp) * q)] for q in [0.25, 0.5, 0.75]]spline_design = truncated_power_basis(horsepower_values, quantiles, degree=3)spline_beta = normal_equation_solve(spline_design, mpg_values)print(f"Regression spline degrees of freedom: {len(spline_beta)}")spline_curve = []for value in x_grid:    row = [1.0]    row.extend(value ** d for d in range(1, 4))    for knot in quantiles:        diff = value - knot        row.append(diff ** 3 if diff > 0 else 0.0)    spline_curve.append(sum(row[i] * spline_beta[i] for i in range(len(spline_beta))))svg_scatter(horsepower_values, mpg_values, filename='partII_c_spline.svg', title='Part II (c): Regression Spline vs Polynomial', xlabel='Horsepower', ylabel='MPG', curves=[(x_grid, best_curve, f"Polynomial degree {best_degree_poly}"), (x_grid, spline_curve, 'Regression spline')])print('Saved spline comparison plot to partII_c_spline.svg')

In [ ]:
# Part II (d)candidate_knots = [1, 2, 3, 4]cv_results_ns = {}ns_models = {}for knot_count in candidate_knots:    sorted_hp = sorted(horsepower_values)    step = len(sorted_hp) // (knot_count + 1)    knots = [sorted_hp[(i + 1) * step] for i in range(knot_count)]    errors = []    for fold in folds:        train_idx = [i for i in range(len(horsepower_values)) if i not in fold]        test_idx = fold        train_x = [horsepower_values[i] for i in train_idx]        train_y = [mpg_values[i] for i in train_idx]        train_design = natural_cubic_spline_basis(train_x, knots)        beta = normal_equation_solve(train_design, train_y)        for idx in test_idx:            value = horsepower_values[idx]            row = natural_cubic_spline_basis([value], knots)[0]            pred = sum(row[i] * beta[i] for i in range(len(beta)))            errors.append((mpg_values[idx] - pred) ** 2)    cv_results_ns[knot_count] = sum(errors) / len(errors)    ns_models[knot_count] = (knots, normal_equation_solve(natural_cubic_spline_basis(horsepower_values, knots), mpg_values))for knot_count in candidate_knots:    print(f"Natural spline with {knot_count} interior knots: CV error = {cv_results_ns[knot_count]:.4f}")best_knots_count = min(cv_results_ns, key=cv_results_ns.get)best_knots, best_ns_beta = ns_models[best_knots_count]ns_curve = []for value in x_grid:    row = natural_cubic_spline_basis([value], best_knots)[0]    ns_curve.append(sum(row[i] * best_ns_beta[i] for i in range(len(best_ns_beta))))print(f"Best natural spline uses {best_knots_count} interior knots with {len(best_ns_beta)} coefficients.")svg_scatter(horsepower_values, mpg_values, filename='partII_d_natural_spline.svg', title='Part II (d): Natural Spline vs Polynomial', xlabel='Horsepower', ylabel='MPG', curves=[(x_grid, best_curve, f"Polynomial degree {best_degree_poly}"), (x_grid, ns_curve, f"Natural spline ({best_knots_count} knots)")])print('Saved natural spline comparison plot to partII_d_natural_spline.svg')

In [ ]:
# Part II (e)lambda_candidates = [0.01, 0.1, 1.0]sample_indices = list(range(0, len(horsepower_values), 2))smooth_x_sample = [horsepower_values[i] for i in sample_indices]smooth_y_sample = [mpg_values[i] for i in sample_indices]folds_smooth = k_fold_indices(len(smooth_x_sample), 3, seed=99)cv_smoothing = {}for lam in lambda_candidates:    errors = []    for fold in folds_smooth:        train_idx = [i for i in range(len(smooth_x_sample)) if i not in fold]        test_idx = fold        train_x = [smooth_x_sample[i] for i in train_idx]        train_y = [smooth_y_sample[i] for i in train_idx]        fitted = smoothing_spline(train_x, train_y, lam)        paired = sorted(zip(train_x, fitted))        xs = [p[0] for p in paired]        ys = [p[1] for p in paired]        for idx in test_idx:            value = smooth_x_sample[idx]            if value <= xs[0]:                pred = ys[0]            elif value >= xs[-1]:                pred = ys[-1]            else:                pred = ys[0]                for j in range(1, len(xs)):                    if xs[j] >= value:                        x0, y0 = xs[j - 1], ys[j - 1]                        x1, y1 = xs[j], ys[j]                        ratio = (value - x0) / (x1 - x0) if x1 != x0 else 0.0                        pred = y0 + ratio * (y1 - y0)                        break            errors.append((smooth_y_sample[idx] - pred) ** 2)    cv_smoothing[lam] = sum(errors) / len(errors)for lam in lambda_candidates:    print(f"Smoothing spline lambda {lam}: CV error = {cv_smoothing[lam]:.4f}")best_lambda_smooth = min(cv_smoothing, key=cv_smoothing.get)print(f"Best smoothing spline lambda: {best_lambda_smooth}")smooth_curve = smoothing_spline_predict(horsepower_values, mpg_values, best_lambda_smooth, x_grid)svg_scatter(horsepower_values, mpg_values, filename='partII_e_smoothing_spline.svg', title='Part II (e): Smoothing Spline vs Polynomial', xlabel='Horsepower', ylabel='MPG', curves=[(x_grid, best_curve, f"Polynomial degree {best_degree_poly}"), (x_grid, smooth_curve, f"Smoothing spline (lambda {best_lambda_smooth})")])print('Saved smoothing spline comparison plot to partII_e_smoothing_spline.svg')

In [ ]:
# Part II (f)cylinders = [row['cylinders'] for row in auto_data]years = [row['year'] for row in auto_data]X_gam = {    'cylinders': cylinders,    'horsepower': horsepower_values,    'year': years,}gam_result = backfitting_gam(X_gam, mpg_values, lam=1.0, max_iter=50)print(f"GAM intercept: {gam_result['intercept']:.4f}")print(f"Residual standard deviation: {gam_result['sigma']:.4f}")for name, values in gam_result['functions'].items():    avg = sum(values) / len(values)    print(f"Component {name}: mean {avg:.4f}, range [{min(values):.4f}, {max(values):.4f}]")svg_line_plot(cylinders, [(gam_result['functions']['cylinders'], 'Effect', '0')], 'partII_f_gam_cylinders.svg', 'GAM: Cylinders effect', 'Cylinders', 'Partial Effect')svg_line_plot(horsepower_values, [(gam_result['functions']['horsepower'], 'Effect', '0')], 'partII_f_gam_horsepower.svg', 'GAM: Horsepower effect', 'Horsepower', 'Partial Effect')svg_line_plot(years, [(gam_result['functions']['year'], 'Effect', '0')], 'partII_f_gam_year.svg', 'GAM: Year effect', 'Year', 'Partial Effect')print('Saved GAM component plots with approximate standard errors (uniform bands not shown due to simplification).')

In [ ]:
# Part II (g)filtered_indices = [i for i, row in enumerate(auto_data) if row['cylinders'] not in (3, 5)]filtered_data = [auto_data[i] for i in filtered_indices]filtered_mpg = [mpg_values[i] for i in filtered_indices]filtered_cylinders = [auto_data[i]['cylinders'] for i in filtered_indices]filtered_horsepower = [horsepower_values[i] for i in filtered_indices]filtered_year = [auto_data[i]['year'] for i in filtered_indices]X_gam_filtered = {    'cylinders': filtered_cylinders,    'horsepower': filtered_horsepower,    'year': filtered_year,}gam_filtered = backfitting_gam(X_gam_filtered, filtered_mpg, lam=1.0, max_iter=50)print(f"Filtered GAM intercept: {gam_filtered['intercept']:.4f}")print(f"Filtered GAM residual std: {gam_filtered['sigma']:.4f}")svg_line_plot(filtered_cylinders, [(gam_filtered['functions']['cylinders'], 'Effect', '0')], 'partII_g_gam_cylinders.svg', 'Filtered GAM: Cylinders', 'Cylinders', 'Partial Effect')svg_line_plot(filtered_horsepower, [(gam_filtered['functions']['horsepower'], 'Effect', '0')], 'partII_g_gam_horsepower.svg', 'Filtered GAM: Horsepower', 'Horsepower', 'Partial Effect')svg_line_plot(filtered_year, [(gam_filtered['functions']['year'], 'Effect', '0')], 'partII_g_gam_year.svg', 'Filtered GAM: Year', 'Year', 'Partial Effect')print('Saved filtered GAM component plots.')

In [ ]:
# Part II (h)print('Model comparison summary:')print(f"  Polynomial degree {best_degree_poly}: CV error {cv_errors_poly[best_degree_poly]:.4f}")print(f"  Natural spline ({best_knots_count} knots): CV error {cv_results_ns[best_knots_count]:.4f}")print(f"  Smoothing spline (lambda {best_lambda_smooth}): CV error {cv_smoothing[best_lambda_smooth]:.4f}")print(f"  GAM (full data) residual std: {gam_result['sigma']:.4f}")print(f"  GAM (filtered data) residual std: {gam_filtered['sigma']:.4f}")print('These metrics highlight trade-offs among flexibility, interpretability, and prediction accuracy across models.')